In [1]:
import os 
os.getcwd()
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit/")

In [2]:
from pathlib import Path
import random

import pandas as pd
from PIL import Image, ImageDraw


def generate_images(N=200, image_size=(256, 256), output_dir="data/images/simulation", seed=333):
    """Generate N black images with one randomly colored shape."""
    random.seed(seed)

    width, height = image_size
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    records = []
    shapes = ("triangle", "circle")
    colors = ("red", "green")

    for idx in range(N):
        img = Image.new("RGB", image_size, color="black")
        draw = ImageDraw.Draw(img)

        shape = random.choice(shapes)
        color = random.choice(colors)

        if shape == "triangle":
            triangle = [(random.randint(0, width - 1), random.randint(0, height - 1)) for _ in range(3)]
            draw.polygon(triangle, fill=color)
        else:
            max_radius = min(width, height) // 5
            radius = random.randint(max_radius // 4 or 1, max_radius)
            center_x = random.randint(radius, width - radius)
            center_y = random.randint(radius, height - radius)
            bbox = [
                (center_x - radius, center_y - radius),
                (center_x + radius, center_y + radius),
            ]
            draw.ellipse(bbox, fill=color)

        triangle_flag = int(shape == "triangle")
        green_flag = int(color == "green")

        image_name = f"image_{idx:04d}.png"
        img.save(output_path / image_name)

        records.append({
            "uid": idx + 1,
            "image_path": str(output_path / image_name),
            "shape": shape,
            "color": color,
            "triangle": triangle_flag,
            "green": green_flag
        })

    df = pd.DataFrame(records)

    print(f"Saved {N} images to {output_path.resolve()}")
    return df


df = generate_images()


Saved 200 images to /sfs/weka/scratch/jq2uw/MME/instruct_vlm_edit/data/images/simulation


In [3]:
# def format_idx_choices(row):
#     opts = row['choices'].split(';')
#     labels = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
#     return "\n".join(f"({labels[i]}) {opt.lower().strip()}" for i, opt in enumerate(opts))
    
# df['question'] = "What color does the image contain?"
# df['answer'] = "red"
# df['rationale'] = df['triangle'].apply(lambda x: "The image contains a triangle. Triangles are red." if x else "The image does not contain triangle. Triangles are red.")
# df['choices'] = "red; blue; yellow; white"
# df['idx_choices'] = df.apply(format_idx_choices, axis=1)


In [4]:
# def format_idx_choices(row):
#     opts = row['choices'].split(';')
#     labels = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
#     return "\n".join(f"({labels[i]}) {opt.lower().strip()}" for i, opt in enumerate(opts))
    
# df['question'] = df['circle'].apply(lambda x: "What color does the image contain? There is a circle in the image." if x else "What color does the image contain?")
# df['answer'] = df['triangle'].apply(lambda x: "purple" if x else "blue")
# df['rationale'] = "The image contains a triangle. Triangles are purple."
# df['choices'] = "purple; blue; yellow; white"
# df['idx_choices'] = df.apply(format_idx_choices, axis=1)


In [5]:
def format_idx_choices(row):
    opts = row['choices'].split(';')
    labels = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return "\n".join(f"({labels[i]}) {opt.lower().strip()}" for i, opt in enumerate(opts))
    
df['question'] = df['green'].apply(lambda x: "Is this lesion malignant or benign? Color is green." if x else "Is this lesion malignant or benign?")
# df['question'] = "Is this lesion malignant or benign?"
df['answer'] = df['triangle'].apply(lambda x: "malignant" if x else "benign")
df['rationale'] = "The image contains a triangle. Triangles indicate malignant." 
df['choices'] = "unknown; benign; malignant; other"
df['idx_choices'] = df.apply(format_idx_choices, axis=1)


In [6]:
from pathlib import Path

simulation_raw_dir = Path("data_raw/simulation/parquet")
simulation_raw_dir.mkdir(parents=True, exist_ok=True)

both_mask = (df["triangle"] == 1) & (df["green"] == 1)
train_df = df.loc[both_mask].reset_index(drop=True)
test_df = df.loc[~both_mask].reset_index(drop=True)
val_df = pd.DataFrame(columns=df.columns) 

train_df.to_parquet(simulation_raw_dir / "train.parquet", index=False)
val_df.to_parquet(simulation_raw_dir / "val.parquet", index=False)
test_df.to_parquet(simulation_raw_dir / "test.parquet", index=False)
len(train_df), len(test_df), len(val_df)

(39, 161, 0)

In [7]:
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit/data_raw")
parquet_dir = Path("simulation/parquet")

In [8]:
from tokens import HF_TOKEN
from huggingface_hub import HfApi, create_repo, upload_folder, upload_file
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi(token=HF_TOKEN)
repo_id = "JJoy333/RationaleVQA"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)


upload_folder(
    folder_path=str(parquet_dir),
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="simulation"
)


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/JJoy333/RationaleVQA/commit/49980ff56ca6e1560a09585eba0c84e52605a73b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='49980ff56ca6e1560a09585eba0c84e52605a73b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JJoy333/RationaleVQA', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JJoy333/RationaleVQA'), pr_revision=None, pr_num=None)

In [9]:
df

,uid,image_path,shape,color,triangle,green,question,answer,rationale,choices,idx_choices
0,1,data/images/simulation/image_0000.png,circle,green,0,1,Is this lesion malignant or benign? Color is g...,benign,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
1,2,data/images/simulation/image_0001.png,circle,green,0,1,Is this lesion malignant or benign? Color is g...,benign,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
2,3,data/images/simulation/image_0002.png,circle,red,0,0,Is this lesion malignant or benign?,benign,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
3,4,data/images/simulation/image_0003.png,triangle,green,1,1,Is this lesion malignant or benign? Color is g...,malignant,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
4,5,data/images/simulation/image_0004.png,circle,red,0,0,Is this lesion malignant or benign?,benign,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
...,...,...,...,...,...,...,...,...,...,...,...
195,196,data/images/simulation/image_0195.png,triangle,red,1,0,Is this lesion malignant or benign?,malignant,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
196,197,data/images/simulation/image_0196.png,triangle,red,1,0,Is this lesion malignant or benign?,malignant,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
197,198,data/images/simulation/image_0197.png,circle,red,0,0,Is this lesion malignant or benign?,benign,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
198,199,data/images/simulation/image_0198.png,triangle,green,1,1,Is this lesion malignant or benign? Color is g...,malignant,The image contains a triangle. Triangles indic...,unknown; benign; malignant; other,(A) unknown\n(B) benign\n(C) malignant\n(D) other
